# Reproducing Neupane, Fiete and Jazayeri (2024) from dandiset 000897

**Paper.** Neupane S, Fiete I, Jazayeri M (2024). Mental navigation in the primate entorhinal
cortex. *Nature* 630, 704-711. [doi:10.1038/s41586-024-07557-z](https://doi.org/10.1038/s41586-024-07557-z).
The claims file (`claims/000897/10.1038-s41586-024-07557-z.yaml`) was written from the version of
record in Europe PMC (PMC11224022): main text, Methods and Extended Data legends, with the
correction of 4 July 2024, and the paper's code (github.com/jazlab/Neupane_Fiete_Jazayeri_mental_navigation,
commit 4bee109).

**Dandiset.** [DANDI:000897](https://dandiarchive.org/dandiset/000897/0.240605.1710), version
0.240605.1710 (doi:10.48324/dandi.000897/0.240605.1710). The paper links the dandiset without a
version; this is its only published version, and the draft lists the same 30 files with the same
total size. Only the `*_behavior+ecephys.nwb` file of each session is read (trials, sorted units,
eye and joystick traces); the `*_ecephys.nwb` files hold the raw voltage.

**What the dandiset holds.** 15 sessions (monkey A 6, monkey M 9) of the paper's 32 (A 17, M 15),
with 364 and 729 units against the paper's 614 and 864. Proportions, not counts, can therefore be
compared with the paper's.

| Claim | Tested here |
|---|---|
| fig1bc-produced-matches-actual | tested (recorded sessions only) |
| fig1e-immediate-generalization | blocked: the generalization sessions are not in the dandiset |
| fig2-periodic-proportion | tested (proportions) |
| fig2d-periodicity-at-065 | tested |
| ed3-no-eye-hand-periodicity | tested in monkey A only; eye and joystick clocks differ from the trial clock in 10 of 15 sessions |
| ed2b-pi-navigation-above-iti | blocked: the ITI cannot be located in the files |
| ed2c-pi-error-below-correct | tested |
| fig2e-first-peak-near-065 | tested |
| fig2f-earlier-peaks-multiples | tested |
| ed4d-overshoot-longer-period | tested |
| fig3b-onset-and-offset-coding | tested |
| fig3d-offset-coding-stronger | tested |
| ed4e-rate-codes-distance-over-slope | tested |
| ed4fg-distance-over-landmark | tested |
| fig3e-xcorr-conserved-iti | blocked: the ITI cannot be located in the files |
| ed5-xcorr-conserved-other-epochs | blocked: depends on fig3e |
| fig4h-periodic-fano-factor | tested (proportions) |
| ed8-fano-periodic-without-rate-periodicity | tested |
| ed10-motor-preparation-signal | tested (proportions) |
| fig4g-reset-model-fits-better | blocked: not attempted in the run's time |
| fig4b-learned-landmark-input | blocked: a model simulation, not attempted in the run's time |
| fig4d-model-subscalar-variability | blocked: a model simulation, not attempted in the run's time |
| ed6d-model-fano-periodic | blocked: a model simulation, not attempted in the run's time |
| ed5efg-tighter-phase-clustering | blocked: depends on fig3e and fig4d |

Methods come from the claims file, and details it lacks from the paper, cited in `mnav.py` and
in each section. The notebook takes about 17 minutes, most of it the PI of every unit, the
surrogate nulls and the ANOVAs.

In [ ]:
import time
import warnings

import numpy as np
import pandas as pd
import statsmodels.api as sm
import statsmodels.formula.api as smf
from scipy import stats

import mnav

pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 40)
pd.set_option("display.max_rows", 300)
warnings.filterwarnings("ignore")

started = time.time()
sessions = mnav.load_all()
print(f"loaded {len(sessions)} sessions from dandiset {mnav.DANDISET_ID} version {mnav.DANDISET_VERSION} "
      f"in {time.time() - started:.0f} s")

## Data check

Each check of the metadata against the values is stated under its output.

In [ ]:
rows = []
for s in sessions:
    tr = s["trials"]
    q = s["units"].get("quality", s["units"].get("KSLabel"))
    rows.append({"monkey": s["monkey"], "session": s["session"], "units": len(s["spikes"]),
                 "label_column": "quality" if "quality" in s["units"] else "KSLabel",
                 **{f"label_{k}": int(v) for k, v in q.value_counts().items()},
                 "trials": len(tr), "mnav": int((tr.trial_type == 3).sum()),
                 "nts_type1": int((tr.trial_type == 1).sum()), "type2": int((tr.trial_type == 2).sum()),
                 "seqq": "/".join(f"{int(k)}:{v}" for k, v in tr.seqq.value_counts().sort_index().items()),
                 "attempt": "/".join(f"{int(k)}:{v}" for k, v in tr.attempt.value_counts().sort_index().items())})
info = pd.DataFrame(rows).fillna(0)
print(info.to_string(index=False))
print("\nSessions per monkey:", info.groupby("monkey").size().to_dict(), "(paper: A 17, M 15)")
print("Units per monkey, all labels:", info.groupby("monkey").units.sum().to_dict(), "(paper: A 614, M 864)")
noise = info.get("label_noise", pd.Series(0, index=info.index))
print("Units per monkey without those labelled noise:", (info.units - noise).groupby(info.monkey).sum().to_dict())

The dandiset holds 15 sessions (A 6, M 9) of the paper's 32 (A 17, M 15), with 364 units in
monkey A and 729 in M, against 614 and 864 in the paper. The monkey A files label units with
phy's `quality` (good, mua, noise, unsorted); 103 of the 364 are labelled noise, which the
paper's "single units and multi-units" would not include. The monkey M files carry only
Kilosort's `KSLabel` (good, mua). Units are analysed here as the files hold them, all labels
included; the count without noise units is given as a variant where it matters.

In [ ]:
rows = []
for s in sessions:
    tr = s["trials"]
    m = (tr.trial_type == 3) & (tr.attempt <= 2)
    d = tr[m]
    joy = d.joy1offttl - d.joy1onttl
    by_seq = {}
    for q, g in d.groupby("seqq"):
        by_seq[f"ta/0.65==target-curr, seqq {int(q)}"] = round(float(np.mean(np.isclose(g.ta / 0.65, g.target - g.curr))), 3)
    sp_all = np.concatenate([x for x in s["spikes"] if len(x)])
    rows.append({"monkey": s["monkey"], "session": s["session"],
                 "ta_values": len(np.unique(d.ta)), "ta_range": f"{d.ta.min():.2f}..{d.ta.max():.2f}",
                 "|joy off-on| - |tp| p1,p99 (ms)": f"{1000 * np.percentile(joy - d.tp.abs(), 1):.0f},{1000 * np.percentile(joy - d.tp.abs(), 99):.0f}",
                 "sign(tp)==sign(ta)": round(float(np.mean(np.sign(d.tp) == np.sign(d.ta))), 3),
                 **by_seq,
                 "trials_s": f"{tr.start_time.min():.0f}..{tr.stop_time.max():.0f}",
                 "spikes_s": f"{sp_all.min():.0f}..{sp_all.max():.0f}",
                 "eye_ts_s": f"{s['info']['eye_ts_first']:.0f}..{s['info']['eye_ts_last']:.0f}",
                 "hand_ts_s": f"{s['info']['hand_ts_first']:.0f}..{s['info']['hand_ts_last']:.0f}"})
print(pd.DataFrame(rows).to_string(index=False))
print("\nDistances (|ta|/0.65) of MNAV trials, all sessions:",
      pd.concat([s["trials"][s["trials"].trial_type == 3].distance for s in sessions]).value_counts().sort_index().to_dict())
for m in ("A", "M"):
    t3 = pd.concat([s["trials"] for s in sessions if s["monkey"] == m])
    t3 = t3[t3.trial_type == 3]
    print(f"monkey {m}, |ta| values (s) per seqq:", {int(q): sorted(np.unique(np.round(g.ta.abs(), 3)).tolist()) for q, g in t3.groupby("seqq")})
print("tp == 99 or 999 (no production) rows:", sum(int((s["trials"].tp.abs() >= 99).sum()) for s in sessions),
      "; attempt codes 99/999:", sum(int((s["trials"].attempt >= 99).sum()) for s in sessions))

The trial table's `ta` takes the ten values +-0.65 to +-3.25 s, as the six-landmark line with
0.65 s spacing has, and the joystick offset minus onset equals |`tp`| within about 20 ms, so
`tp` is the produced vector in seconds, signed as `ta`. In monkey A `ta`/0.65 equals `target` -
`curr` on every trial. In monkey M, about half of the MNAV trials have `seqq` 3, whose vectors
come in steps of 0.975 s (0.975 to 4.875 s, printed above), not the 0.65 s the paper describes for
both monkeys; the paper and the files do not describe this sequence. Analyses that use distance
keep `seqq` < 3, as the paper's code does for the PI (claims, opening comment).
Rows with `attempt` 99 or 999 have `tp` 99 or 999 (no production) and are not used.

Spike times and trial times share one range in every session. The eye and joystick timestamps do
not in several sessions: they start hundreds of thousands of seconds away from the trial times
(A 08152019; M 04122021, 04132021, 04142021, 04212021, 04222021, 04252021, 05062021, 05072021), so
in those sessions the behavioural traces are on another clock, which the files do not describe.
The eye and joystick traces are used only for claim ed3, in the sessions whose timestamps span the
trial times.

In [ ]:
# Clock check: the paper's motor response before joystick offset (Extended Data Fig. 10). For each
# session, the fraction of units whose mean rate differs (Welch t-test, p < 0.01) between 0.4-0.1 s
# and 1.0-0.6 s before joystick offset on first-attempt MNAV trials, with the true offset times and
# with the offset times of each trial replaced by those of a trial 5 rows later (a shifted label).
rows = []
for s in sessions:
    tr = s["trials"]
    d = tr[(tr.trial_type == 3) & (tr.attempt == 1) & (tr.tp.abs() > 1.5)]
    off = d.joy1offttl.values
    out = {"monkey": s["monkey"], "session": s["session"], "trials": len(d)}
    for name, t in (("true", off), ("shifted", np.roll(off, 5) + 7.0)):
        sig = 0
        for sp in s["spikes"]:
            late = mnav.mean_rate(sp, t - 0.4, t - 0.1)
            early = mnav.mean_rate(sp, t - 1.0, t - 0.6)
            if np.std(late - early) > 0 and stats.ttest_rel(late, early).pvalue < 0.01:
                sig += 1
        out[f"frac_units_{name}"] = round(sig / len(s["spikes"]), 3)
    rows.append(out)
clock = pd.DataFrame(rows)
print(clock.to_string(index=False))

In 12 of the 15 sessions more units change their rate before the true joystick offset than
before the shifted times (printed above), which is consistent with spikes and trials sharing a
clock there. In three sessions (A 09172019, M 04132021, M 04142021) no unit does so with either
timing, and while preparing this notebook the population rate of these sessions was flat (within
a few percent) around the stimulus onset, go cue, joystick onset and joystick offset (not run
here). In M 04132021 the spikes also stop at 9,306 s while trials run to 18,599 s. For these three
sessions a shared clock could not be established; they are kept, as the paper analysed all units,
and results are broken down by session where it matters.

## fig1bc-produced-matches-actual

> In MNAV the produced vectors match the actual vectors in direction and magnitude; performance is
> quantified by the slope of the regression of v_p on v_a.

Criterion (claims file): per recorded session, OLS of signed v_p on signed v_a over all MNAV
trials with a recorded production (code: 0.01 < |tp| and tp < 99 s); reproduced when, in each
monkey, the median session slope is at least 0.8 and every slope is positive. Variant: the
`validtrials_mm` trials only.

In [ ]:
rows = []
for s in sessions:
    tr = s["trials"]
    d = tr[(tr.trial_type == 3) & (tr.tp.abs() > 0.01) & (tr.tp < 99)]
    v = d[d.validtrials_mm == 1]
    fit = stats.linregress(d.ta, d.tp)
    fitv = stats.linregress(v.ta, v.tp)
    rows.append({"monkey": s["monkey"], "session": s["session"], "trials": len(d), "slope_all": fit.slope,
                 "r_all": fit.rvalue, "valid_trials": len(v), "slope_valid": fitv.slope})
fig1 = pd.DataFrame(rows)
print(fig1.round(3).to_string(index=False))
print("\nPer monkey:")
print(fig1.groupby("monkey").agg(sessions=("slope_all", "size"), median_slope_all=("slope_all", "median"),
                                 min_slope_all=("slope_all", "min"), max_slope_all=("slope_all", "max"),
                                 median_slope_valid=("slope_valid", "median")).round(3).to_string())
for m, g in fig1.groupby("monkey"):
    print(f"monkey {m}: median slope >= 0.8: {g.slope_all.median() >= 0.8}; all slopes positive: {(g.slope_all > 0).all()}")

**Result: reproduced.** The median session slope is 0.872 in monkey A (6 sessions, 0.825 to
0.946) and 0.860 in monkey M (9 sessions, 0.687 to 0.893), so at least 0.8 in both, and every
slope is positive. One monkey M session (04142021) is below 0.8. On the `validtrials_mm` trials
the medians are 0.936 and 0.869. These are the recorded sessions only, a subset of the
behavioural sessions of Fig. 1c.

## The periodicity index (PI) of every unit

Computed once, as the claims file's opening comment and `mnav.py` describe, for both lag ranges
(Methods 0 to 1,300 ms, code 100 to 1,600 ms), per unit and direction, on first-attempt MNAV
trials over at least three landmarks with |tp| > 1 s and `seqq` < 3.

In [ ]:
t = time.time()
pit = mnav.periodicity_table(sessions)
print(f"PI curves for {len(pit)} unit-direction pairs in {time.time() - t:.0f} s")
print("Units with more than 15 kept trials, per monkey and direction:")
print(pit[pit.k > mnav.MIN_TRIALS].groupby(["monkey", "direction"]).size().to_string())
print("Trials in the PI per session and direction (median over sessions):",
      pit.drop_duplicates(["session", "direction"]).groupby(["monkey", "direction"]).n_trials.median().to_dict())
print("Monkey M trials with seqq 3 left out by seqq < 3:",
      sum(int(((s["trials"].trial_type == 3) & (s["trials"].seqq == 3)).sum()) for s in sessions if s["monkey"] == "M"))

In [ ]:
# Null distributions (Methods: GP surrogate, Poisson, smoothed as the data; 1,000 repeats).
rng_null = np.random.default_rng(897)
nulls = {}
t = time.time()
for monkey in ("A", "M"):
    sub = pit[(pit.monkey == monkey)]
    groups = sub.drop_duplicates(["session", "direction"])
    all_durations = np.concatenate(groups.durations.values)
    rates = []
    for s in sessions:
        if s["monkey"] != monkey:
            continue
        trs = mnav.pi_trials(s["trials"])
        t0, t1 = mnav.nav_windows(trs)
        rates += [float(np.mean(mnav.mean_rate(sp, t0, t1))) for sp in s["spikes"]]
    rate = float(np.median(rates))
    bank, bank_d = mnav.surrogate_bank(rng_null, all_durations, rate, 4000)
    k_med = int(np.median(sub[sub.k > mnav.MIN_TRIALS].k))
    d_med = float(np.median(all_durations))
    for lagname, lags in (("methods", mnav.LAGS_METHODS), ("code", mnav.LAGS_CODE)):
        nulls[(monkey, "global", lagname)] = mnav.null_from_bank(rng_null, bank, bank_d, [d_med] * k_med, lags)
        for _, g in groups.iterrows():
            nulls[(monkey, g.session, g.direction, lagname)] = mnav.null_from_bank(rng_null, bank, bank_d, g.durations, lags)
    print(f"monkey {monkey}: surrogate rate {rate:.2f} spikes/s (median unit rate), global null with {k_med} trials "
          f"of {d_med:.0f} ms; bank of {len(bank)} surrogate trials")
print(f"nulls in {time.time() - t:.0f} s")

In [ ]:
# Classification under every reading. Rules: (a1) score = max PI - mean PI from the first
# positive lag on, above the maximum over lags of the surrogate PI mean + 2 SD; (a2) the same score
# above the surrogate scores' mean + 2 SD; (b) PI at any lag above the surrogate mean + 2 SD at that
# lag. Null durations: own (the session and direction's trials) or global. Lags: methods or code.
def thresholds(null):
    m, sd = null.mean(0), null.std(0)
    scores = np.array([mnav.pi_score(p) for p in null])
    return {"per_lag": m + 2 * sd, "a1": float((m + 2 * sd).max()), "a2": float(scores.mean() + 2 * scores.std())}

thr = {key: thresholds(v) for key, v in nulls.items()}
READINGS = [(rule, dur, lag) for rule in ("a1", "a2", "b") for dur in ("own", "global") for lag in ("methods", "code")]

def classify(row, rule, dur, lag):
    pi = getattr(row, "pi_" + lag)
    if pi is None or row.k <= mnav.MIN_TRIALS:
        return False
    key = (row.monkey, "global", lag) if dur == "global" else (row.monkey, row.session, row.direction, lag)
    th = thr[key]
    if rule == "b":
        return bool(np.any(pi > th["per_lag"]))
    return mnav.pi_score(pi) > th[rule]

for rule, dur, lag in READINGS:
    pit[f"per_{rule}_{dur}_{lag}"] = [classify(r, rule, dur, lag) for r in pit.itertuples()]
for lag in ("methods", "code"):
    lags = mnav.LAGS_METHODS if lag == "methods" else mnav.LAGS_CODE
    pit[f"period_{lag}"] = [np.nan if p is None else lags[int(np.argmax(p))] / 1000 for p in pit["pi_" + lag]]
    pit[f"score_{lag}"] = [np.nan if p is None else mnav.pi_score(p) for p in pit["pi_" + lag]]
    i650 = int(np.nonzero(lags == 650)[0][0])
    pit[f"pi650_{lag}"] = [np.nan if p is None else p[i650] for p in pit["pi_" + lag]]
MAIN = ("a2", "own", "methods")
main_col = "per_" + "_".join(MAIN)
i650 = int(np.nonzero(mnav.LAGS_METHODS == 650)[0][0])
base = pit[["monkey", "session", "unit", "direction", "k", "pi_methods", "period_methods", main_col]].copy()
base["above650"] = [False if p is None else p[i650] > thr[(r.monkey, r.session, r.direction, "methods")]["per_lag"][i650]
                    for p, r in zip(base.pi_methods, base.itertuples())]
print("Thresholds, global null:")
for key in [k for k in thr if k[1] == "global"]:
    print(" ", key, {k: round(v, 3) for k, v in thr[key].items() if k != "per_lag"},
          "per-lag max", round(float(thr[key]["per_lag"].max()), 3))
print("Neuron scores (methods lags), median and 90th percentile per monkey:",
      pit[pit.k > mnav.MIN_TRIALS].groupby("monkey").score_methods.quantile([0.5, 0.9]).round(3).to_dict())

## fig2-periodic-proportion

> A substantial fraction of EC neurons have significantly periodic activity during mental
> navigation, 37% (231/614) in monkey A and 36% (311/864) in monkey M, and the proportion per
> session ranged from 0% to 45% depending on recording site.

A unit counts as periodic when it is periodic in either direction; the denominator is all units of
the monkey. The dandiset holds 6 of the 17 monkey A sessions and 9 of the 15 monkey M sessions, so
the counts cannot reach 231 and 311; they are compared as proportions of the units the dandiset
holds (231/614 and 311/864, each with the claims file's 10% relative tolerance), and the raw counts
are printed.

In [ ]:
units_total = info.groupby("monkey").units.sum()
noise_by_monkey = (noise).groupby(info.monkey).sum()
paper = {"A": (231, 614), "M": (311, 864)}
labels = pit.drop_duplicates(["monkey", "session", "unit"])[["monkey", "session", "unit", "label"]]
rows, sess_rows = [], []
for rule, dur, lag in READINGS:
    col = f"per_{rule}_{dur}_{lag}"
    per_unit = pit.groupby(["monkey", "session", "unit"])[col].any().reset_index()
    out = {"reading": f"{rule}/{dur}/{lag}"}
    for m in ("A", "M"):
        n = int(per_unit[per_unit.monkey == m][col].sum())
        p_ref = paper[m][0] / paper[m][1]
        out[f"{m}_count"] = n
        out[f"{m}_pct"] = round(100 * n / units_total[m], 1)
        nn = per_unit[(per_unit.monkey == m)].merge(labels, on=["monkey", "session", "unit"])
        nn = nn[nn.label != "noise"]
        out[f"{m}_pct_no_noise"] = round(100 * nn[col].mean(), 1)
        out[f"{m}_in_tol"] = abs(n / units_total[m] - p_ref) <= 0.1 * p_ref
    ses = per_unit.groupby("session")[col].mean() * 100
    out["session_pct_min"] = round(ses.min(), 1)
    out["session_pct_max"] = round(ses.max(), 1)
    out["sessions_in_0_49.5"] = bool((ses <= 49.5).all())
    out["conditions_met"] = int(out["A_in_tol"]) + int(out["M_in_tol"]) + int(out["sessions_in_0_49.5"])
    rows.append(out)
    sess_rows.append(ses.rename(f"{rule}/{dur}/{lag}"))
fig2 = pd.DataFrame(rows)
print("Paper: A 231/614 = 37.6%, M 311/864 = 36.0%; tolerance 10% of each proportion (A 33.9-41.4%, M 32.4-39.6%).")
print(fig2.to_string(index=False))
print("\nPer-session percentage of periodic units under each reading:")
print(pd.concat(sess_rows, axis=1).round(1).T.to_string())

**Result: not reproduced.** Under no reading is either monkey's proportion within 10% of the
paper's (A 37.6%, M 36.0%). The readings that use the code's score with a threshold from the
surrogate scores (a2) give 9.1% to 24.2% in monkey A and 8.2% to 17.0% in monkey M; the code's
score against the per-lag surrogate threshold (a1) gives at most 4.9% and 0.8%; the Methods'
rule of PI above the surrogate at any lag (b) gives 78.0% to 81.3% and 58.0% to 60.6%. Only the
per-session range is met, under the a1 and a2 readings (session maxima 7.4% to 34.7%, within
0 to 49.5%); under b the session maxima are 87.5% to 95.3%. At most one of the three conditions
holds under any reading, so the claim is not reproduced under any of them. Without the units
labelled noise, reading a2/own/methods gives 28.0% in monkey A. The null's conversion of the GP
into a firing rate is an assumption (`mnav.py`), and the counts are judged as proportions because
the dandiset holds 15 of the 32 sessions.

## fig2d-periodicity-at-065

> Across neurons with significant PI, the lag of the peak PI was at or near 0.65 s, the temporal
> distance between landmarks.

Criterion: median periodicity (lag of the maximum PI) of the periodic units within 0.60 to 0.70 s
in every monkey and direction; the mode in 50 ms bins is reported as well.

In [ ]:
rows = []
for rule, dur, lag in READINGS:
    col = f"per_{rule}_{dur}_{lag}"
    for (m, d), g in pit[pit[col]].groupby(["monkey", "direction"]):
        p = g[f"period_{lag}"]
        hist, edges = np.histogram(p, bins=np.arange(0, 1.651, 0.05))
        rows.append({"reading": f"{rule}/{dur}/{lag}", "monkey": m, "direction": int(d), "n": len(p),
                     "median_s": round(float(p.median()), 3), "iqr_s": f"{p.quantile(.25):.2f}-{p.quantile(.75):.2f}",
                     "mode_bin_s": f"{edges[hist.argmax()]:.2f}-{edges[hist.argmax() + 1]:.2f}",
                     "frac_0.6_0.7": round(float(((p >= 0.6) & (p <= 0.7)).mean()), 3),
                     "in_0.60_0.70": 0.60 <= p.median() <= 0.70})
fig2d = pd.DataFrame(rows)
print(fig2d.to_string(index=False))
print("\nGroups (monkey x direction) with median in 0.60-0.70 s, per reading:")
print(fig2d.groupby("reading")["in_0.60_0.70"].agg(["sum", "size"]).to_string())
main_sel = pit[pit[main_col]]
print("\nMain reading (a2/own/methods), all periodic unit-directions pooled: median periodicity",
      round(float(main_sel.period_methods.median()), 3), "s; per monkey:", main_sel.groupby("monkey").period_methods.median().round(3).to_dict())
allp = pit[pit.k > mnav.MIN_TRIALS]
print("\nReference: median periodicity of all units with >15 trials (methods lags):",
      allp.groupby(["monkey", "direction"]).period_methods.median().round(3).to_dict())

**Result: partially reproduced, resting on an assumption.** Under readings a2 (the code's score
against the surrogate scores) the median periodicity of the periodic units is within 0.60 to
0.70 s in 3 (own-duration null, Methods lags; the main reading) or 4 (code lags) of the 4 monkey
and direction groups; with the global null and Methods lags in 2 of 4. The exception under the
main reading is monkey A, direction -1 (median 0.590 s). Under reading a1 too few units are
periodic (1 to 15 per group) and the medians are 0.52 to 0.63 s; under reading b, which selects
most units, the medians are 0.72 to 0.95 s, close to those of all units (0.73 to 0.91 s). The
readings disagree; reading a2 is adopted, as it compares the code's score with the same score
on the surrogates. The modal 50 ms bin is 0.55 to 0.70 s under a2.

## ed2c-pi-error-below-correct

> Neurons significantly periodic at 0.65 s had a lower PI on error trials than on correct trials,
> suggesting weaker periodicity contributed to larger errors.

PI measure (claims, ed2b and ed2c): PI at the 650 ms lag minus the mean PI from the first lag at
which PI is positive. Correct trials are the PI trials; error trials are the first attempt of
trials with more than one attempt (`attempt` == 2; Methods, MNAV), with the same distance, |tp|
and `seqq` rules; the code's variant takes `succ` == 0 trials with |tp| > 1.8 s. Neurons, two
readings: (i) code, periodic units whose PI at 650 ms exceeds the surrogate PI mean + 2 SD at
650 ms; (ii) Results, periodic units with periodicity in 0.60 to 0.70 s. Periodic units are those
of the main reading (rule a2, own-duration null, Methods lags; see the fig2 claims' conclusions).
Both conditions need more than 15 trials. One-tailed paired t-test, correct > error.

In [ ]:
def pi650_measure(pi, lags=mnav.LAGS_METHODS):
    i650 = int(np.nonzero(lags == 650)[0][0])
    pos = np.nonzero(pi > 0)[0]
    base = pi[pos[0]:].mean() if len(pos) else pi.mean()
    return float(pi[i650] - base)

t = time.time()
pit_err = mnav.periodicity_table(sessions, attempt=2, lag_sets={"methods": mnav.LAGS_METHODS})
pit_err_code = mnav.periodicity_table(
    sessions, attempt=2, lag_sets={"methods": mnav.LAGS_METHODS},
    extra=lambda tr: (tr.succ == 0) & (tr.tp.abs() > 1.8))
print(f"error-trial PI in {time.time() - t:.0f} s")
rows = []
for err_name, err in (("attempt2", pit_err), ("code_succ0_tp1.8", pit_err_code)):
    mrg = base.merge(err[["monkey", "session", "unit", "direction", "k", "pi_methods"]],
                     on=["monkey", "session", "unit", "direction"], suffixes=("", "_err"))
    for sel_name, sel in (("i_code_pi650", mrg[main_col] & mrg.above650),
                          ("ii_results_0.6-0.7", mrg[main_col] & mrg.period_methods.between(0.6, 0.7))):
        g = mrg[sel & (mrg.k_err > mnav.MIN_TRIALS)]
        for (m, d), gg in g.groupby(["monkey", "direction"]):
            c = np.array([pi650_measure(p) for p in gg.pi_methods])
            e = np.array([pi650_measure(p) for p in gg.pi_methods_err])
            tt = stats.ttest_rel(c, e, alternative="greater") if len(gg) > 1 else None
            rows.append({"error_trials": err_name, "neurons": sel_name, "monkey": m, "direction": int(d),
                         "n": len(gg), "dropped_<=15_err_trials": int((sel & (mrg.monkey == m) & (mrg.direction == d)).sum() - len(gg)),
                         "correct_mean": round(c.mean(), 3), "error_mean": round(e.mean(), 3),
                         "t": None if tt is None else round(tt.statistic, 2), "p_one_tailed": None if tt is None else tt.pvalue,
                         "meets_p<1e-4": bool(tt is not None and tt.pvalue < 1e-4 and c.mean() > e.mean())})
ed2c = pd.DataFrame(rows)
print(ed2c.to_string(index=False))
print("\nGroups meeting the criterion per reading (of 4):")
print(ed2c.groupby(["error_trials", "neurons"])["meets_p<1e-4"].sum().to_string())

**Result: partially reproduced, resting on an assumption.** With the Methods' error trials and
the code's neuron selection (PI at 650 ms above the surrogate threshold), correct-trial PI exceeds
error-trial PI at p < 0.0001 in 3 of 4 groups (A -1: t = 5.24, n = 21; A 1: t = 5.21, n = 26;
M 1: t = 4.67, n = 35); in M -1 the difference is in the same direction with p = 0.012 (n = 26).
With the Results' selection (periodicity within 0.60 to 0.70 s) only 1 of 4 groups reaches
p < 0.0001, though all four differ in the same direction (p 5.7e-07 to 0.0095). With the code's
error trials (`succ` 0, |tp| > 1.8 s) all four groups meet it under the code's selection. The
code's selection is adopted, as it tests periodicity at 0.65 s, which is what the claim names.
All groups are smaller than the paper's (68 to 104 neurons), as the dandiset holds about half of
the sessions.

## ed4d-overshoot-longer-period

> For trials of the same desired distance, the periodicity of EC neurons is significantly longer
> (lower frequency) on overshoot trials (top tertile of produced vector) than on undershoot trials
> (bottom tertile).

Periodic units (main reading), per direction in which they are periodic; correct trials of
distance 3 (with the PI rules); per session and direction, trials sorted by |tp| - |ta|, bottom
and top tertiles; periodicity (Methods lags) on each; units with fewer than 15 kept trials in a
tertile dropped, and periodicities of 0 or above 1 s dropped. Rank-sum test of overshoot against
undershoot across units, per monkey; the code's variant lags (100 to 1,090 ms) are also shown.

In [ ]:
def tertile_mask(which):
    def f(tr):
        m = (tr.trial_type == 3) & (tr.attempt == 1) & (tr.distance == 3) & (tr.tp.abs() > 1) & (tr.seqq < 3)
        out = pd.Series(False, index=tr.index)
        for d in (1, -1):
            g = tr[m & (tr.direction == d)]
            err = g.tp.abs() - g.ta.abs()
            lo, hi = err.quantile(1 / 3), err.quantile(2 / 3)
            out[g.index[err <= lo] if which == "under" else g.index[err >= hi]] = True
        return out
    return f

lagsets = {"methods": mnav.LAGS_METHODS, "code": np.arange(100, 1091, 10)}
t = time.time()
under = mnav.periodicity_table(sessions, extra=tertile_mask("under"), lag_sets=lagsets)
over = mnav.periodicity_table(sessions, extra=tertile_mask("over"), lag_sets=lagsets)
print(f"tertile PI in {time.time() - t:.0f} s")
rows = []
for lag, lags in lagsets.items():
    mrg = under.merge(over, on=["monkey", "session", "unit", "direction"], suffixes=("_u", "_o"))
    mrg = mrg.merge(base[["monkey", "session", "unit", "direction", main_col]], on=["monkey", "session", "unit", "direction"])
    mrg = mrg[mrg[main_col]]
    n_periodic = mrg.groupby("monkey").size()
    ok = (mrg.k_u >= 15) & (mrg.k_o >= 15) & mrg[f"pi_{lag}_u"].notna() & mrg[f"pi_{lag}_o"].notna()
    mrg = mrg[ok].copy()
    mrg["per_u"] = [lags[int(np.argmax(p))] / 1000 for p in mrg[f"pi_{lag}_u"]]
    mrg["per_o"] = [lags[int(np.argmax(p))] / 1000 for p in mrg[f"pi_{lag}_o"]]
    keep = (mrg.per_u > 0) & (mrg.per_u <= 1) & (mrg.per_o > 0) & (mrg.per_o <= 1)
    for m in ("A", "M"):
        g = mrg[keep & (mrg.monkey == m)]
        if len(g) < 2:
            rows.append({"lags": lag, "monkey": m, "n": len(g)})
            continue
        rs = stats.ranksums(g.per_o, g.per_u)
        rows.append({"lags": lag, "monkey": m, "periodic_unit_dirs": int(n_periodic.get(m, 0)),
                     "kept_>=15_trials": int((mrg.monkey == m).sum()), "n": len(g),
                     "median_under_s": g.per_u.median(), "median_over_s": g.per_o.median(),
                     "mean_diff_over-under_s": round((g.per_o - g.per_u).mean(), 4),
                     "ranksum_Z(over-under)": round(rs.statistic, 2), "p": rs.pvalue,
                     "meets(over>under,p<0.05)": bool(rs.pvalue < 0.05 and g.per_o.median() >= g.per_u.median() and rs.statistic > 0)})
ed4d = pd.DataFrame(rows)
print(ed4d.to_string(index=False))
print("Trials per tertile per session and direction (median):",
      int(np.median([len(d) for d in under.drop_duplicates(['session', 'direction']).durations])))

**Result: partially reproduced.** In monkey M the overshoot tertile has a longer periodicity
(median 0.69 against 0.61 s, rank-sum Z = 3.13, p = 0.0017, n = 53 units-directions); in monkey A
the difference is in the same direction but not significant (0.59 against 0.57 s, Z = 1.31,
p = 0.19, n = 47). The code's lag range gives the same. Of the periodic unit-directions, 50 of
107 (A) and 66 of 152 (M) had at least 15 trials in both tertiles.

## ed10-motor-preparation-signal

> Most EC neurons show a motor-related response before joystick offset, 77% (477/614) in monkey A
> and 70% (607/864) in monkey M, including most non-periodic neurons (81%, 368/452, in A and 89%,
> 432/481, in M).

Criterion (claims, from the code): direction 1, first-attempt MNAV trials of distance at least 3
with |tp| > 1.5 s, at least 6 trials; spike-count rate in tp - 1 to tp - 0.6 s and in tp - 0.4 to
tp - 0.1 s after joystick onset; 200 bootstrap means of each; two-sample t-test at p < 0.0001
(two-sided; increase only as a variant). Non-periodic: not periodic (main reading) with more than
15 PI trials. As for fig2, the dandiset holds a subset of the sessions, so the counts are compared
as proportions of the units it holds; the raw counts are printed. Variant: either direction.

In [ ]:
rng10 = np.random.default_rng(10)
rows = []
for s in sessions:
    tr = s["trials"]
    for d in (1, -1):
        g = tr[(tr.trial_type == 3) & (tr.attempt == 1) & (tr.distance >= 3) & (tr.tp.abs() > 1.5) & (tr.seqq < 3) & (tr.direction == d)]
        on, tp = g.joy1onttl.values, g.tp.abs().values
        for u, sp in enumerate(s["spikes"]):
            row = {"monkey": s["monkey"], "session": s["session"], "unit": u, "direction": d, "n": len(g)}
            if len(g) >= 6:
                early = mnav.mean_rate(sp, on + tp - 1.0, on + tp - 0.6)
                late = mnav.mean_rate(sp, on + tp - 0.4, on + tp - 0.1)
                idx = rng10.integers(0, len(g), size=(200, len(g)))
                be, bl = early[idx].mean(1), late[idx].mean(1)
                p = stats.ttest_ind(bl, be).pvalue if (be.std() + bl.std()) > 0 else 1.0
                row.update(p=p, increase=bl.mean() > be.mean())
            rows.append(row)
mot = pd.DataFrame(rows)
mot["motor"] = mot.p.fillna(1) < 1e-4
mot["motor_increase"] = mot.motor & mot.increase.fillna(False).astype(bool)
per_dir = mot.merge(base[["monkey", "session", "unit", "direction", "k", main_col]], on=["monkey", "session", "unit", "direction"])
unit_periodic = base.groupby(["monkey", "session", "unit"])[main_col].any()
unit_k = base.groupby(["monkey", "session", "unit"]).k.max()
paper10 = {"A": dict(motor=(477, 614), nonper_motor=(368, 452), nonper=(452, 614)),
           "M": dict(motor=(607, 864), nonper_motor=(432, 481), nonper=(481, 864))}
rows = []
for variant, sel in (("direction 1 (code)", per_dir.direction == 1), ("either direction", per_dir.direction != 0)):
    for col in ("motor", "motor_increase"):
        u = per_dir[sel].groupby(["monkey", "session", "unit"])[col].any()
        df = pd.DataFrame({"motor": u, "periodic": unit_periodic, "k": unit_k}).reset_index()
        df["nonper"] = (~df.periodic) & (df.k > mnav.MIN_TRIALS)
        for m in ("A", "M"):
            g = df[df.monkey == m]
            n_all, n_mot = len(g), int(g.motor.sum())
            n_np, n_np_mot = int(g.nonper.sum()), int((g.nonper & g.motor).sum())
            ref = paper10[m]
            rows.append({"variant": variant, "test": col, "monkey": m, "units": n_all, "motor": n_mot,
                         "motor_pct": round(100 * n_mot / n_all, 1), "paper_pct": round(100 * ref["motor"][0] / ref["motor"][1], 1),
                         "nonperiodic": n_np, "nonper_pct_of_units": round(100 * n_np / n_all, 1),
                         "paper_nonper_pct": round(100 * ref["nonper"][0] / ref["nonper"][1], 1),
                         "nonper_motor": n_np_mot, "nonper_motor_pct": round(100 * n_np_mot / max(n_np, 1), 1),
                         "paper_nonper_motor_pct": round(100 * ref["nonper_motor"][0] / ref["nonper_motor"][1], 1)})
ed10 = pd.DataFrame(rows)
print(ed10.to_string(index=False))
main10 = ed10[(ed10.variant == "direction 1 (code)") & (ed10.test == "motor")]
for r in main10.itertuples():
    within = [abs(r.motor_pct - r.paper_pct) <= 0.1 * r.paper_pct,
              abs(r.nonper_pct_of_units - r.paper_nonper_pct) <= 0.1 * r.paper_nonper_pct,
              abs(r.nonper_motor_pct - r.paper_nonper_motor_pct) <= 0.1 * r.paper_nonper_motor_pct]
    print(f"monkey {r.monkey}: within 10% of the paper's proportion (motor, non-periodic, motor among non-periodic): {within}; "
          f"motor and motor-among-non-periodic above 50%: {r.motor_pct > 50}, {r.nonper_motor_pct > 50}")
print("\nMotor units (direction 1, two-sided) per session (%):")
print((per_dir[per_dir.direction == 1].groupby(["monkey", "session"]).motor.mean() * 100).round(1).to_string())

**Result: partially reproduced.** Under the code's test (two-sided, direction 1) 75.5% of the
monkey A units (275/364) and 67.9% of the monkey M units (495/729) are motor-related, within 10%
of the paper's 77.7% and 70.3%; among non-periodic units 85.6% (178/208) and 85.5% (272/318),
within 10% of 81.4% and 89.8%. The share of non-periodic units, 57.1% and 43.6% of all units
against the paper's 73.6% and 55.7%, is not, as it follows from the periodic proportion
(fig2-periodic-proportion). So not all six values are within tolerance, while the four
proportions are each above 50%. Counting increases only, 33.0% and 33.1% of units are
motor-related. The bootstrap t-test compares 200 resampled means, so its p-value shrinks with the
number of resamples and not with the data; this is the paper's procedure, followed as written.

## Ramping and distance coding: shared selection

Correct (first-attempt) MNAV trials with `seqq` < 3 (distance from `ta` in 0.65 s steps).
Units with at least 10 such trials per distance (1 to 5) in both directions (claims, fig3b and
fig3d, from the code); direction 1 (ta > 0) is analysed (code line 93).

In [ ]:
def coding_trials(tr):
    return tr[(tr.trial_type == 3) & (tr.attempt == 1) & (tr.seqq < 3) & (tr.tp.abs() > 0.01) & (tr.tp < 99)]

sel_rows = []
for s in sessions:
    g = coding_trials(s["trials"])
    cnt = g.groupby(["direction", "distance"]).size()
    ok = all(cnt.get((d, k), 0) >= 10 for d in (1, -1) for k in range(1, 6))
    sel_rows.append({"monkey": s["monkey"], "session": s["session"], "min_trials_per_dir_dist": int(cnt.min()), "eligible": ok,
                     "units": len(s["spikes"])})
elig = pd.DataFrame(sel_rows)
print(elig.to_string(index=False))
print("Eligible units per monkey:", elig[elig.eligible].groupby("monkey").units.sum().to_dict())

## fig3b-onset-and-offset-coding

> Single EC neurons encode temporal distance in their firing rate before joystick onset, before
> joystick offset, or, in a small number, at both.

Cross-validated R2 (Methods, Cross-validated R2; claims): mean rate in the window, each distance
subsampled to the smallest count, random halves, linear fit of rate on distance 1 to 5 on the
training half scored on the held-out half, 50 splits averaged; positive coding is mean R2 > 0;
units with R2 below -1 dropped. Windows: (a) 100 ms before onset and offset, (b) 200 ms before.
Null: the same with the distances shuffled.

In [ ]:
rng3b = np.random.default_rng(3)

def cv_r2(y, x, rng, n_split=50, shuffle=False):
    out = []
    for _ in range(n_split):
        xx = rng.permutation(x) if shuffle else x
        perm = rng.permutation(len(y))
        tr_i, te_i = perm[: len(y) // 2], perm[len(y) // 2:]
        b = np.polyfit(xx[tr_i], y[tr_i], 1)
        pred = np.polyval(b, xx[te_i])
        yt = y[te_i]
        ss = np.sum((yt - yt.mean()) ** 2)
        out.append(1 - np.sum((yt - pred) ** 2) / ss if ss > 0 else np.nan)
    return np.nanmean(out) if np.any(np.isfinite(out)) else np.nan

rows = []
for s in sessions:
    if not elig.set_index("session").eligible[s["session"]]:
        continue
    g = coding_trials(s["trials"])
    g = g[g.direction == 1]
    nmin = g.groupby("distance").size().loc[1:5].min()
    g = pd.concat([gg.sample(nmin, random_state=int(rng3b.integers(1e9))) for k, gg in g.groupby("distance") if 1 <= k <= 5])
    x = g.distance.values.astype(float)
    on, off = g.joy1onttl.values, g.joy1offttl.values
    for u, sp in enumerate(s["spikes"]):
        row = {"monkey": s["monkey"], "session": s["session"], "unit": u}
        for w in (0.1, 0.2):
            for ev, t in (("onset", on), ("offset", off)):
                y = mnav.mean_rate(sp, t - w, t)
                row[f"r2_{ev}_{int(w * 1000)}"] = cv_r2(y, x, rng3b)
                row[f"r2shuf_{ev}_{int(w * 1000)}"] = cv_r2(y, x, rng3b, shuffle=True)
        rows.append(row)
r2 = pd.DataFrame(rows)
res = []
for w in (100, 200):
    for m, g in r2.groupby("monkey"):
        on, off = g[f"r2_onset_{w}"], g[f"r2_offset_{w}"]
        valid = (on >= -1) & (off >= -1)
        on, off = on[valid], off[valid]
        son, soff = g[f"r2shuf_onset_{w}"][valid], g[f"r2shuf_offset_{w}"][valid]
        p_on_only = float(((on > 0) & (off <= 0)).mean())
        p_off_only = float(((off > 0) & (on <= 0)).mean())
        p_both = float(((on > 0) & (off > 0)).mean())
        p_shuf = float(((son > 0) | (soff > 0)).mean())
        p_shuf_on, p_shuf_off = float((son > 0).mean()), float((soff > 0).mean())
        res.append({"window_ms": w, "monkey": m, "units": int(valid.sum()), "dropped_r2<-1": int((~valid).sum()),
                    "onset_only": round(p_on_only, 3), "offset_only": round(p_off_only, 3), "both": round(p_both, 3),
                    "shuffled_onset>0": round(p_shuf_on, 3), "shuffled_offset>0": round(p_shuf_off, 3),
                    "meets": bool(p_on_only > p_shuf_on and p_off_only > p_shuf_off and p_both < p_on_only and p_both < p_off_only)})
fig3b = pd.DataFrame(res)
print(fig3b.to_string(index=False))
print("\nShuffled-regressor proportions are compared with the coding proportions at the same epoch (onset only against shuffled onset, offset only against shuffled offset).")

**Result: reproduced.** In both monkeys and both windows, the proportions of units coding
distance at onset only (A 0.7% and 1.0%, M 8.0% and 8.2% for 100 and 200 ms) and at offset only
(A 8.3% and 6.0%, M 2.3% and 3.6%) exceed the proportion with positive cross-validated R2 under
the shuffled regressor (0 in every case), and fewer units code at both (A 0%, M 1.1% and 2.9%)
than at either alone. The shuffled proportions are 0 because R2 averaged over 50 held-out halves
is always negative for a shuffled regressor, so this comparison is weak; the coding proportions
themselves are small. Units with R2 below -1 were dropped (A 76 and 63, M 201 and 185). Monkey M
sessions with fewer than 10 trials per distance in a direction (3 of 9) are not used.

## fig3d-offset-coding-stronger

> At the population level distance is encoded weakly before joystick onset and robustly and
> linearly before joystick offset; offset R2 exceeds onset R2 in both monkeys.

Targeted dimensionality reduction (Methods; claims): pseudo-population of each monkey's eligible
units, direction 1. Per resample, each unit's trials of each distance are split at random into
halves. Rates in 20 ms bins (a standard choice for the time resolution of the regression, which
the Methods do not give) in the windows -200 to 0 ms before onset and -500 to -300 ms before
offset (Methods); centred per unit by its training mean across both windows; per bin, the
regression r = beta * dist across training trials; the axis is the coefficient vector at the bin
of largest norm; held-out trials of each distance are averaged, averaged over the window,
centred, and projected on the axis; R2 of a linear fit of the five projections on distance; 50
resamples. Variant: the coefficients projected onto the principal components holding 80% of the
training variance (the code's denoising). Two-sided rank-sum test of offset against onset R2.

In [ ]:
rng3d = np.random.default_rng(33)
BIN = 0.02
win = {"onset": (-0.2, 0.0), "offset": (-0.5, -0.3)}
pop = {"A": [], "M": []}
for s in sessions:
    if not elig.set_index("session").eligible[s["session"]]:
        continue
    g = coding_trials(s["trials"])
    g = g[(g.direction == 1) & g.distance.between(1, 5)]
    for u, sp in enumerate(s["spikes"]):
        unit = {}
        for ev, col in (("onset", "joy1onttl"), ("offset", "joy1offttl")):
            a, b = win[ev]
            edges = np.arange(a, b + 1e-9, BIN)
            t = g[col].values
            unit[ev] = np.stack([mnav.mean_rate(sp, t + e0, t + e0 + BIN) for e0 in edges[:-1]], axis=1)  # trials x bins
        unit["dist"] = g.distance.values.astype(int)
        pop[s["monkey"]].append(unit)

def tdr_once(units, rng, denoise=False):
    train = {"onset": [], "offset": []}
    test = {"onset": [], "offset": []}
    for unit in units:
        tr_idx, te_idx = [], []
        for k in range(1, 6):
            idx = rng.permutation(np.nonzero(unit["dist"] == k)[0])
            tr_idx.append(idx[: len(idx) // 2])
            te_idx.append(idx[len(idx) // 2:])
        mu = np.mean([unit[ev][np.concatenate(tr_idx)].mean() for ev in ("onset", "offset")])
        for ev in ("onset", "offset"):
            X = unit[ev] - mu
            train[ev].append([X[i] for i in tr_idx])
            test[ev].append(np.array([X[i].mean() for i in te_idx]))   # per distance, averaged over bins and trials
    out = {}
    for ev in ("onset", "offset"):
        nb = train[ev][0][0].shape[1]
        beta = np.zeros((len(units), nb))
        cond = np.zeros((len(units), 5 * nb))
        for i, per_d in enumerate(train[ev]):
            num = sum((k + 1) * per_d[k].sum(0) for k in range(5))
            den = sum((k + 1) ** 2 * len(per_d[k]) for k in range(5))
            beta[i] = num / den
            cond[i] = np.concatenate([per_d[k].mean(0) for k in range(5)])
        if denoise:
            c = cond - cond.mean(1, keepdims=True)
            U, S, _ = np.linalg.svd(c, full_matrices=False)
            npc = int(np.searchsorted(np.cumsum(S ** 2) / np.sum(S ** 2), 0.8) + 1)
            beta = U[:, :npc] @ (U[:, :npc].T @ beta)
        tstar = int(np.argmax(np.linalg.norm(beta, axis=0)))
        axis = beta[:, tstar] / np.linalg.norm(beta[:, tstar])
        proj = np.array(test[ev]).T @ axis       # 5 distances
        fit = stats.linregress(np.arange(1, 6), proj)
        out[ev] = fit.rvalue ** 2
    return out

rows = []
for denoise in (False, True):
    for m in ("A", "M"):
        if not pop[m]:
            continue
        rs = [tdr_once(pop[m], rng3d, denoise) for _ in range(50)]
        on = np.array([r["onset"] for r in rs])
        off = np.array([r["offset"] for r in rs])
        test = stats.ranksums(on, off)
        rows.append({"denoise_80pct_PCs": denoise, "monkey": m, "units": len(pop[m]),
                     "onset_R2_median": round(float(np.median(on)), 3), "offset_R2_median": round(float(np.median(off)), 3),
                     "onset_R2_iqr": f"{np.percentile(on, 25):.2f}-{np.percentile(on, 75):.2f}",
                     "offset_R2_iqr": f"{np.percentile(off, 25):.2f}-{np.percentile(off, 75):.2f}",
                     "ranksum_Z(onset-offset)": round(test.statistic, 2), "p": test.pvalue,
                     "meets(offset>onset,p<1e-4)": bool(np.median(off) > np.median(on) and test.pvalue < 1e-4)})
fig3d = pd.DataFrame(rows)
print(fig3d.to_string(index=False))

**Result: partially reproduced.** In monkey A the offset R2 exceeds the onset R2 (medians 0.854
against 0.709, rank-sum Z = -5.14, p = 2.8e-07; with denoising 0.853 against 0.568, p = 5.2e-14).
In monkey M the order is reversed: onset R2 exceeds offset R2 (0.787 against 0.684, Z = 6.85,
p = 7.3e-12; with denoising the same). The bin width of the regression (20 ms) is a choice the
Methods do not make.

## ed4fg-distance-over-landmark and ed4e-rate-codes-distance-over-slope: the ANOVAs

Four-way ANOVA (distance, start landmark, target landmark, direction; Methods), per unit with at
least 10 correct trials in each of the 30 start and target conditions (claims, from the code's
landmark ANOVA), on the mean rate (spike count over the window) and on the slope of a line fitted
to the smoothed rate in the window. Type II sums of squares (each factor adjusted for the others;
the Methods do not say which, a standard choice). Units with zero variance in a window are skipped.

In [ ]:
def anova_F(df, y):
    model = smf.ols(f"{y} ~ C(distance) + C(curr) + C(target) + C(direction)", data=df).fit()
    a = sm.stats.anova_lm(model, typ=2)
    return {f: (a.loc[f"C({f})", "F"], a.loc[f"C({f})", "PR(>F)"]) for f in ("distance", "curr", "target", "direction")}

def slope_of(sp, t0, t1):
    rates = mnav.smoothed_rates(sp, t0, t1)
    return np.array([np.polyfit(np.arange(len(r)) / 1000, r, 1)[0] if len(r) > 2 else np.nan for r in rates])

WINDOWS = {"rate_-500_-300": ("rate", -0.5, -0.3), "rate_-200_0": ("rate", -0.2, 0.0), "rate_-400_-300": ("rate", -0.4, -0.3),
           "rate_-550_-250": ("rate", -0.55, -0.25), "slope_-550_-250": ("slope", -0.55, -0.25), "slope_-300_0": ("slope", -0.3, 0.0)}
t = time.time()
rows = []
anova_sessions = []
for s in sessions:
    g = coding_trials(s["trials"])
    cnt = g.groupby(["curr", "target"]).size()
    if len(cnt) < 30 or cnt.min() < 10:
        anova_sessions.append({"session": s["session"], "monkey": s["monkey"], "conditions": len(cnt), "min_trials": int(cnt.min()), "used": False})
        continue
    anova_sessions.append({"session": s["session"], "monkey": s["monkey"], "conditions": len(cnt), "min_trials": int(cnt.min()), "used": True})
    off = g.joy1offttl.values
    for u, sp in enumerate(s["spikes"]):
        row = {"monkey": s["monkey"], "session": s["session"], "unit": u}
        df = g[["distance", "curr", "target", "direction"]].copy()
        for name, (kind, a, b) in WINDOWS.items():
            y = mnav.mean_rate(sp, off + a, off + b) if kind == "rate" else slope_of(sp, off + a, off + b)
            if np.nanstd(y) == 0:
                continue
            df["y"] = y
            res_ = anova_F(df.dropna(), "y")
            for f, (F, p) in res_.items():
                row[f"{name}_{f}_F"] = F
                row[f"{name}_{f}_p"] = p
        rows.append(row)
anova = pd.DataFrame(rows)
print(pd.DataFrame(anova_sessions).to_string(index=False))
print(f"ANOVAs for {len(anova)} units in {time.time() - t:.0f} s; units per monkey:", anova.groupby("monkey").size().to_dict())

## ed4fg-distance-over-landmark

> Significantly less variance of firing rate near joystick offset is explained by the start or the
> target landmark than by distance.

Units with a significant effect (p < 0.05) of distance or of the landmark; distance F against
landmark F, by (a) a t-test on the differences (Methods; paired) and (b) a two-sample t-test
(legend). Windows (a) -500 to -300 ms (Methods), (b) the last 200 ms (legend); the code's -400 to
-300 ms is a variant.

In [ ]:
rows = []
for wname in ("rate_-500_-300", "rate_-200_0", "rate_-400_-300"):
    for m, g in anova.groupby("monkey"):
        for lm in ("curr", "target"):
            cols = [f"{wname}_distance_F", f"{wname}_{lm}_F", f"{wname}_distance_p", f"{wname}_{lm}_p"]
            gg = g.dropna(subset=cols)
            sig = gg[(gg[cols[2]] < 0.05) | (gg[cols[3]] < 0.05)]
            fd, fl = sig[cols[0]], sig[cols[1]]
            tp_ = stats.ttest_1samp(fd - fl, 0) if len(sig) > 1 else None
            t2 = stats.ttest_ind(fd, fl) if len(sig) > 1 else None
            rows.append({"window": wname, "monkey": m, "landmark": "start" if lm == "curr" else "target", "units_tested": len(gg),
                         "units_sig_either": len(sig), "mean_F_distance": round(fd.mean(), 2), "mean_F_landmark": round(fl.mean(), 2),
                         "paired_t": None if tp_ is None else round(tp_.statistic, 2), "paired_p": None if tp_ is None else tp_.pvalue,
                         "two_sample_t": None if t2 is None else round(t2.statistic, 2), "two_sample_p": None if t2 is None else t2.pvalue,
                         "meets_paired": bool(tp_ is not None and tp_.pvalue < 1e-4 and fd.mean() > fl.mean()),
                         "meets_two_sample": bool(t2 is not None and t2.pvalue < 1e-4 and fd.mean() > fl.mean())})
ed4fg = pd.DataFrame(rows)
print(ed4fg.to_string(index=False))
main_fg = ed4fg[ed4fg.window != "rate_-400_-300"]
print("\nMain readings (windows a, b; tests a, b): comparisons meeting p < 1e-4 with distance F larger:",
      int(main_fg.meets_paired.sum() + main_fg.meets_two_sample.sum()), "of", 2 * len(main_fg))

**Result: partially reproduced.** Distance F exceeds landmark F in every comparison (mean F 4.2
to 8.2 against 1.3 to 4.2), but p < 0.0001 is reached in 8 of the 16 main comparisons (2 windows,
2 tests, 2 landmarks, 2 monkeys). In the last 200 ms (legend window) with the t-test on
differences (Methods), all four monkey and landmark comparisons meet it (p 6.8e-05 to 3.8e-17);
in -500 to -300 ms (Methods window) only monkey M's start landmark does (p 1.4e-04 to 4.5e-03
for the others). Monkey M sessions with fewer than 10 trials in a condition (4 of 9) are not used.

## ed4e-rate-codes-distance-over-slope

> Near joystick offset the distance effect on the mean firing rate is significantly larger than
> that on the slope of the firing rate.

Distance F of the rate ANOVA against distance F of the slope ANOVA across units, two-sample t-test
(legend; the code's paired test is a variant). Window readings: (a) Methods, rate -500 to -300 ms
and slope -550 to -250 ms; (b) legend, rate in the last 200 ms and slope in the last 300 ms; (c)
Results, both in -550 to -250 ms.

In [ ]:
rows = []
for reading, rw, sw in (("a_methods", "rate_-500_-300", "slope_-550_-250"), ("b_legend", "rate_-200_0", "slope_-300_0"),
                        ("c_results", "rate_-550_-250", "slope_-550_-250")):
    for m, g in anova.groupby("monkey"):
        gg = g.dropna(subset=[f"{rw}_distance_F", f"{sw}_distance_F"])
        fr, fs = gg[f"{rw}_distance_F"], gg[f"{sw}_distance_F"]
        t2 = stats.ttest_ind(fr, fs)
        tp_ = stats.ttest_rel(fr, fs)
        rows.append({"reading": reading, "monkey": m, "units": len(gg), "mean_F_rate": round(fr.mean(), 2), "mean_F_slope": round(fs.mean(), 2),
                     "median_F_rate": round(fr.median(), 2), "median_F_slope": round(fs.median(), 2),
                     "two_sample_t": round(t2.statistic, 2), "two_sample_p": t2.pvalue, "paired_t": round(tp_.statistic, 2), "paired_p": tp_.pvalue,
                     "meets(two-sample)": bool(t2.pvalue < 1e-4 and fr.mean() > fs.mean())})
ed4e = pd.DataFrame(rows)
print(ed4e.to_string(index=False))

**Result: not reproduced.** The two-sample t-test finds rate F larger than slope F at
p < 0.0001 only in monkey M with the legend's windows (mean F 3.27 against 1.86, p = 1.9e-05).
Under the Methods' and the Results' windows neither monkey reaches it (p 0.003 to 0.47), and in
monkey A with the legend's windows slope F is the larger (2.77 against 2.29, p = 0.21). The
median F is about 1 in every case. No monkey meets the criterion under all readings.

## The inter-trial interval (claims ed2b, fig3e, ed5)

The paper compares the navigation epoch with ITI segments longer than 4 s (Methods, NTS). The
trial table has no ITI or fixation event: each row starts at the stimulus onset and stops at the
next row's start. The only interval that can hold the ITI is from the joystick offset to the next
trial's start, printed here.

In [ ]:
rows = []
for s in sessions:
    tr = s["trials"]
    nxt = tr.start_time.shift(-1)
    a1 = tr[(tr.trial_type == 3) & (tr.attempt == 1)]
    gap = (a1.stop_time - a1.joy1offttl).dropna()
    rows.append({"monkey": s["monkey"], "session": s["session"],
                 "stop==next_start": round(float(np.mean(np.isclose(tr.stop_time, nxt)[:-1])), 4),
                 "offset_to_next_start_median_s": round(gap.median(), 2), "p99_s": round(gap.quantile(0.99), 2),
                 "n_longer_than_4s": int((gap > 4).sum()), "n_trials": len(gap)})
iti = pd.DataFrame(rows)
print(iti.to_string(index=False))
ab = []
for s in sessions:
    tr = s["trials"]
    a = tr[tr.attempt >= 99]
    ab.append({"monkey": s["monkey"], "session": s["session"], "aborts": len(a),
               "abort_to_next_start_median_s": round(float((a.stop_time - a[["gocuettl", "joy1onttl"]].max(axis=1)).median()), 2)})
print("\nAborted trials (attempt 99 or 999): time from the joystick press to the next trial's start")
print(pd.DataFrame(ab).to_string(index=False))

Every row ends where the next begins, and the time from a completed trial's joystick offset to the
next trial's start is about 0.9 s (A) or 1.2 s (M), longer than 4 s in 0 to 6 trials per session.
After aborted trials (the Methods add a 5 s time-out) the next trial starts a median 6.1 to 6.3 s after
the abort in monkey M sessions and 2.3 to 2.4 s in monkey A (printed for each session above); the paper
does not say that its ITI segments came from there. The ITI segments longer than 4 s that claims
ed2b and fig3e use cannot be located in the files, so those claims, and ed5 and ed5efg, which
depend on fig3e, are not tested.

## fig4h-periodic-fano-factor

> Many EC neurons have a Fano factor that is periodic at about 0.65 s: 55% (341/614) in monkey A
> and 43% (378/864) in monkey M.

Extended Data Fig. 8 legend and claims (from the code): correct MNAV trials with |tp| >= 2.3 s
(`seqq` < 3), per direction, at least 12 trials; spike counts in 200 ms windows stepped by 30 ms
from joystick onset. The end of the time course is not given; it is taken here as 500 ms before
the shortest trial's offset, as for the PI (an assumption). Fano factor across trials,
bootstrapped 100 times (trials resampled); Poisson null from the unit's mean count time course,
100 times; each Fano time course linearly detrended, autocorrelated to 1.6 s, and its PI computed
as for the rate, in 30 ms steps (650 ms is read at the nearest step, 660 ms). Selection readings:
(a) legend, maximum PI in the 300 to 900 ms lag window above the null by a two-sample t-test at
p < 0.0001/20; (b) code, PI at 650 ms against the null by a one-tailed Welch t-test at 0.01.
Direction readings: (i) either direction, (ii) direction 1. Periodicity: lag of the maximum mean
bootstrap PI over 100 to 1,600 ms. Counts are compared as proportions of the units the dandiset
holds, as for fig2.

In [ ]:
rng4h = np.random.default_rng(48)
STEP, WIN = 0.03, 0.2
max_lag = int(round(1.6 / STEP))
flags = np.arange(0, max_lag + 1)
i660 = int(round(0.65 / STEP))
win_lo, win_hi = int(np.ceil(0.3 / STEP)), int(np.floor(0.9 / STEP))

def series_pi(x):
    """PI curve (lags 0..1.6 s in 30 ms steps) of a time course, detrended, ACG normalised at 0."""
    t_ = np.arange(len(x))
    d = x - np.polyval(np.polyfit(t_, x, 1), t_)
    n = len(d)
    nfft = 1 << int(np.ceil(np.log2(2 * n + 2 * max_lag + 2)))
    F = np.fft.rfft(d, nfft)
    ac = np.fft.irfft(F * np.conj(F), nfft)
    if ac[0] <= 0:
        return None
    ac = ac[:max_lag + 1] / ac[0]
    full = np.concatenate([ac[:0:-1], ac])
    return mnav.pi_curve(full, flags)

def fano_curves(counts, rng, n=100, poisson=False):
    out = []
    mu = counts.mean(0)
    for _ in range(n):
        c = rng.poisson(np.broadcast_to(mu, counts.shape)) if poisson else counts[rng.integers(0, len(counts), len(counts))]
        m_, v_ = c.mean(0), c.var(0, ddof=1)
        ff = np.where(m_ > 0, v_ / np.where(m_ > 0, m_, 1), 0.0)
        p = series_pi(ff)
        if p is not None:
            out.append(p)
    return np.array(out)

t = time.time()
rows = []
for s in sessions:
    tr = s["trials"]
    for d in (1, -1):
        g = tr[(tr.trial_type == 3) & (tr.attempt == 1) & (tr.tp.abs() >= 2.3) & (tr.tp.abs() < 99) & (tr.seqq < 3) & (tr.direction == d)]
        if len(g) < 12:
            continue
        on = g.joy1onttl.values
        end = g.tp.abs().min() - mnav.OFFSET_CUT_S
        starts = np.arange(0, end - WIN + 1e-9, STEP)
        for u, sp in enumerate(s["spikes"]):
            counts = np.stack([np.searchsorted(sp, on + a + WIN) - np.searchsorted(sp, on + a) for a in starts], axis=1)
            if counts.sum() == 0:
                continue
            boot = fano_curves(counts, rng4h)
            null = fano_curves(counts, rng4h, poisson=True)
            if len(boot) < 10 or len(null) < 10:
                continue
            bmax = boot[:, win_lo:win_hi + 1].max(1)
            nmax = null[:, win_lo:win_hi + 1].max(1)
            ta_ = stats.ttest_ind(bmax, nmax)
            tb_ = stats.ttest_ind(boot[:, i660], null[:, i660], equal_var=False, alternative="greater")
            mean_pi = boot.mean(0)
            lo = int(np.ceil(0.1 / STEP))
            rows.append({"monkey": s["monkey"], "session": s["session"], "unit": u, "direction": d, "trials": len(g),
                         "sel_a": bool(ta_.pvalue < 1e-4 / 20 and bmax.mean() > nmax.mean()),
                         "sel_b": bool(tb_.pvalue < 0.01),
                         "fano_period": (lo + int(np.argmax(mean_pi[lo:]))) * STEP})
ff = pd.DataFrame(rows)
print(f"Fano factor PI for {len(ff)} unit-direction pairs in {time.time() - t:.0f} s")
paper4h = {"A": (341, 614), "M": (378, 864)}
rows = []
for sel in ("sel_a", "sel_b"):
    for dname, dsel in (("either", ff.direction != 0), ("direction 1", ff.direction == 1)):
        g = ff[dsel]
        per_unit = g.groupby(["monkey", "session", "unit"])[sel].any()
        periods = g[g[sel]].groupby(["monkey", "session", "unit"]).fano_period.first()
        out = {"selection": sel, "direction": dname}
        ok = True
        for m in ("A", "M"):
            n = int(per_unit.loc[m].sum()) if m in per_unit.index.get_level_values(0) else 0
            pct = 100 * n / units_total[m]
            ref = 100 * paper4h[m][0] / paper4h[m][1]
            out[f"{m}_count"] = n
            out[f"{m}_pct"] = round(pct, 1)
            out[f"{m}_paper_pct"] = round(ref, 1)
            out[f"{m}_in_tol"] = abs(pct - ref) <= 0.1 * ref
            pm = periods.loc[m] if m in periods.index.get_level_values(0) else pd.Series(dtype=float)
            out[f"{m}_median_period_s"] = round(float(pm.median()), 3) if len(pm) else None
            ok &= bool(len(pm) and 0.6 <= pm.median() <= 0.7)
        out["medians_in_0.60_0.70"] = ok
        rows.append(out)
fig4h = pd.DataFrame(rows)
print(fig4h.to_string(index=False))
print("\nUnit-direction pairs tested per monkey:", ff.groupby("monkey").size().to_dict())

**Result: partially reproduced.** Under the legend's selection (a) and either direction, the
periodic Fano factor units have median periodicities of 0.69 s (A) and 0.66 s (M), within 0.60
to 0.70 s, but they are 65.4% (238/364) and 52.3% (381/729) of the units, above the paper's
55.5% and 43.8% by more than 10%. With direction 1 only, 44.2% and 32.6%, below it; medians 0.66
and 0.69 s. Under the code's selection (b), monkey A's 59.9% is within tolerance but monkey M's
54.0% is not, and M's median is 0.72 s. No combination puts both proportions within tolerance;
under (a) the medians hold. The end of the Fano time course is an assumption (500 ms before the
shortest trial's offset).

## ed8-fano-periodic-without-rate-periodicity

> The periodic Fano factor is not a by-product of firing-rate periodicity; it is also present in
> neurons whose firing rate is not periodic.

Units not periodic in rate (fig2 main reading, either direction) that have a periodic Fano factor
(fig4h readings). One-sided binomial test of their count against 5% of all units (main) and, as a
variant, 5% of the non-periodic units.

In [ ]:
rate_periodic = base.groupby(["monkey", "session", "unit"])[main_col].any()
rows = []
for sel in ("sel_a", "sel_b"):
    for dname, dsel in (("either", ff.direction != 0), ("direction 1", ff.direction == 1)):
        per_unit = ff[dsel].groupby(["monkey", "session", "unit"])[sel].any()
        df = pd.DataFrame({"fano": per_unit}).join(rate_periodic.rename("rate_periodic"), how="right").fillna(False)
        for m in ("A", "M"):
            g = df.loc[m]
            n_np = int((~g.rate_periodic).sum())
            k = int((g.fano & ~g.rate_periodic).sum())
            p_all = stats.binomtest(k, int(units_total[m]), 0.05, alternative="greater").pvalue
            p_np = stats.binomtest(k, n_np, 0.05, alternative="greater").pvalue
            rows.append({"selection": sel, "direction": dname, "monkey": m, "units": int(units_total[m]), "nonperiodic_rate": n_np,
                         "fano_periodic_among_nonperiodic": k, "binom_p_vs_5pct_all": p_all, "binom_p_vs_5pct_nonperiodic": p_np,
                         "meets_p<0.05": bool(p_all < 0.05)})
ed8 = pd.DataFrame(rows)
print(ed8.to_string(index=False))
print("Code's unlabelled counts, for information: A 99/614, M 74/864.")

**Result: reproduced.** Among units not periodic in rate, 166 of 364 (A) and 287 of 729 (M) have
a periodic Fano factor under the legend's selection and either direction, far above 5% of all
units (binomial p 1.7e-113 and 3.3e-173); every selection and direction reading gives p < 1e-36.
The counts are larger than the code's unlabelled 99/614 and 74/864, as the Fano selection here
admits more units (fig4h).

## fig2e-first-peak-near-065 and fig2f-earlier-peaks-multiples: phase of the local peaks

Methods (Quantification of the phase of local bumps) and claims (from the code): per periodic
unit (main reading) and direction in which it is periodic; first-attempt trials not flagged as
lapses (`validtrials_mm`), distance 4 or 5, `seqq` < 3; trials with no spikes from 1.5 to 0.5 s
before offset dropped (the claims give "low rate" without a threshold; the PI's 0.05 spikes/s is
used, an assumption); at least 25 trials. 100 times, the smoothed rate averaged over a random
third of the trials; the first peak is the highest local maximum (findpeaks, prominence 0.05 on
the min-max normalised window) in the 1 s before offset (Methods), the second in the 1 s before
the first, the third in the 1 s before the second. Null: each trial's spike train circularly
shifted by a random length within the 3.5 s analysed, then the same. Per unit, the KS test of
bootstrap against null first-peak phases and the mode of the bootstrap phases (10 ms bins, a
standard choice). Across units whose KS p < 0.001, the median of the modes; readings for the
KS P << 0.0001: (a) per unit, (b) pooled phases. Variant: the code's -1 to -0.1 s window.

In [ ]:
from scipy.signal import find_peaks

rng2e = np.random.default_rng(265)
SPAN = 3.5

def peak_phases(rate, first_window=(-1.0, 0.0)):
    """Phases (s, negative) of the first three peaks of a rate on the window [-SPAN, 0) before offset."""
    out = []
    hi = first_window[1]
    lo = first_window[0]
    for _ in range(3):
        a, b = int(round((SPAN + lo) * 1000)), int(round((SPAN + hi) * 1000))
        if a < 0 or b - a < 3:
            out.append(np.nan)
            continue
        seg = rate[a:b]
        rngv = seg.max() - seg.min()
        if rngv <= 0:
            out.append(np.nan)
            continue
        x = (seg - seg.min()) / rngv
        pk, _ = find_peaks(x, prominence=0.05)
        if len(pk) == 0:
            out.append(np.nan)
            break
        best = pk[np.argmax(x[pk])]
        phase = (a + best) / 1000 - SPAN
        out.append(phase)
        hi, lo = phase, phase - 1.0
    while len(out) < 3:
        out.append(np.nan)
    return out

def phase_unit(counts, rng, n=100, window=(-1.0, 0.0)):
    k = max(1, len(counts) // 3)
    boot, null = [], []
    for _ in range(n):
        idx = rng.choice(len(counts), k, replace=False)
        r = gaussian(counts[idx].mean(0))
        boot.append(peak_phases(r, window))
        sh = np.stack([np.roll(counts[i], int(rng.integers(0, counts.shape[1]))) for i in idx])
        null.append(peak_phases(gaussian(sh.mean(0)), window))
    return np.array(boot), np.array(null)

from scipy.ndimage import gaussian_filter1d
def gaussian(x):
    return gaussian_filter1d(x, mnav.SMOOTH_SD_MS, truncate=mnav.SMOOTH_TRUNCATE, mode="nearest") * 1000

periodic_pairs = base[base[main_col]][["monkey", "session", "unit", "direction"]]
sess_by = {s["session"]: s for s in sessions}
t = time.time()
rows, pooled = [], {}
for window_name, window in (("methods_-1_0", (-1.0, 0.0)), ("code_-1_-0.1", (-1.0, -0.1))):
    for r in periodic_pairs.itertuples():
        s = sess_by[r.session]
        tr = s["trials"]
        g = tr[(tr.trial_type == 3) & (tr.attempt == 1) & (tr.validtrials_mm == 1) & tr.distance.isin([4, 5])
               & (tr.seqq < 3) & (tr.direction == r.direction)]
        sp = s["spikes"][r.unit]
        off = g.joy1offttl.values
        keep = mnav.mean_rate(sp, off - 1.5, off - 0.5) > mnav.MIN_TRIAL_RATE
        off = off[keep]
        if len(off) < 25:
            rows.append({"window": window_name, "monkey": r.monkey, "session": r.session, "unit": r.unit, "direction": r.direction, "trials": len(off)})
            continue
        counts, _n = mnav.trial_counts(sp, off - SPAN, off)
        boot, null = phase_unit(counts, rng2e, window=window)
        b1, n1 = boot[:, 0][np.isfinite(boot[:, 0])], null[:, 0][np.isfinite(null[:, 0])]
        ks = stats.ks_2samp(b1, n1).pvalue if len(b1) > 1 and len(n1) > 1 else np.nan
        row = {"window": window_name, "monkey": r.monkey, "session": r.session, "unit": r.unit, "direction": r.direction,
               "trials": len(off), "ks_p": ks}
        for j, name in enumerate(("first", "second", "third")):
            v = boot[:, j][np.isfinite(boot[:, j])]
            if len(v):
                h, e = np.histogram(v, bins=np.arange(-SPAN, 0.0001, 0.01))
                row[f"mode_{name}"] = round(e[h.argmax()] + 0.005, 3)
        rows.append(row)
        pooled.setdefault((window_name, r.monkey), []).append((b1, n1))
phase = pd.DataFrame(rows)
print(f"phase analysis in {time.time() - t:.0f} s; periodic unit-directions: {len(periodic_pairs)}; "
      f"with at least 25 trials: {int(phase[phase.window == 'methods_-1_0'].ks_p.notna().sum())}")

In [ ]:
rows = []
for (w, m), g in phase[phase.ks_p.notna()].groupby(["window", "monkey"]):
    sig = g[g.ks_p < 0.001]
    b = np.concatenate([x[0] for x in pooled[(w, m)]])
    nn = np.concatenate([x[1] for x in pooled[(w, m)]])
    pooled_p = stats.ks_2samp(b, nn).pvalue
    rows.append({"window": w, "monkey": m, "units_tested": len(g), "units_ks_p<0.001": len(sig),
                 "units_ks_p<0.0001": int((g.ks_p < 1e-4).sum()), "frac_p<0.0001": round(float((g.ks_p < 1e-4).mean()), 3),
                 "pooled_ks_p": pooled_p, "median_first_mode_s": sig.mode_first.median(),
                 "first_mode_iqr": f"{sig.mode_first.quantile(.25):.2f}..{sig.mode_first.quantile(.75):.2f}",
                 "in_-0.70_-0.60": bool(-0.70 <= sig.mode_first.median() <= -0.60)})
fig2e = pd.DataFrame(rows)
print(fig2e.to_string(index=False))
sig_m = phase[(phase.window == "methods_-1_0") & (phase.ks_p < 0.001)]
print("\nMethods window, both monkeys pooled: median first-peak mode", sig_m.mode_first.median(), "s over", len(sig_m), "units")
print("\nBy direction (Methods window, units with KS p < 0.001):")
print(phase[(phase.window == "methods_-1_0") & (phase.ks_p < 0.001)].groupby(["monkey", "direction"]).mode_first
      .agg(["size", "median"]).to_string())
print("Code's choice of direction for Fig. 2f (A direction -1, M direction 1):",
      phase[(phase.window == "methods_-1_0") & (phase.ks_p < 0.001) &
            (((phase.monkey == "A") & (phase.direction == -1)) | ((phase.monkey == "M") & (phase.direction == 1)))]
      .groupby("monkey").mode_first.median().to_dict())

**Result: partially reproduced.** With the Methods' window, the median first-peak mode of the
units whose KS test gives p < 0.001 is -0.645 s in monkey A (50 units), within -0.70 to -0.60 s,
and -0.735 s in monkey M (56 units), outside it. The KS test separates bootstrap from null in most
units (p < 0.0001 in 44 of 67 in A and 50 of 69 in M), and the pooled test gives p 6.1e-61 and
9.0e-73. The code's window (-1 to -0.1 s) gives -0.675 (A) and -0.805 s (M). By direction, the
medians are -0.67 and -0.595 s (A) and -0.58 and -0.90 s (M). Of 259 periodic unit-directions,
136 had at least 25 trials.

In [ ]:
rows = []
for (w, m), g in phase[(phase.ks_p < 0.001)].groupby(["window", "monkey"]):
    rows.append({"window": w, "monkey": m, "units": len(g),
                 "median_second_s": g.mode_second.median(), "n_second": int(g.mode_second.notna().sum()),
                 "median_third_s": g.mode_third.median(), "n_third": int(g.mode_third.notna().sum()),
                 "second_in_-1.35_-1.25": bool(abs(g.mode_second.median() + 1.30) <= 0.05),
                 "third_in_-2.00_-1.90": bool(abs(g.mode_third.median() + 1.95) <= 0.05)})
fig2f = pd.DataFrame(rows)
print(fig2f.to_string(index=False))
print("\nMethods window, both monkeys pooled: median second-peak mode", sig_m.mode_second.median(),
      "s; median third-peak mode", sig_m.mode_third.median(), "s")

**Result: partially reproduced.** In monkey A the median second-peak mode is -1.315 s (Methods
window), within 0.05 s of -1.30, but the third is -1.585 s, not near -1.95. In monkey M the
second and third are -1.070 and -1.660 s. The second peak meets the criterion in one monkey only.

## ed3-no-eye-hand-periodicity

> Eye position (horizontal and vertical) and joystick position during the navigation epoch show no
> periodicity; their PI at 650 ms is below chance.

Only sessions whose eye and joystick timestamps span the trial times are used (the five monkey A
sessions other than 08152019; see the data check); first the joystick trace is checked against
the trial table's joystick onsets. The signals are treated as the rates are for the PI (claims):
PI trials of both directions pooled, joystick onset to 500 ms before offset, the trace
interpolated to 1 ms and smoothed with the rate kernel, each trial detrended, mean ACG, PI
(Methods lags). Bootstrap readings: (a) 100 resamples of the pooled trials with replacement, full
size; (b) 100 subsamples of a third without replacement. Null: Gaussian process surrogates (SE
kernel, length 100 ms) smoothed as the signals, without the Poisson step (the signals are
continuous), averaged over as many trials as a bootstrap sample, 1,000 times. Chance readings:
(a) the surrogate PI at 650 ms mean + 2 SD; (b) its mean. One-tailed one-sample t-test of the 100
bootstrap PIs at 650 ms against chance.

In [ ]:
from scipy.ndimage import gaussian_filter1d as _g1d
rng3 = np.random.default_rng(3003)
beh_sessions = [s for s in sessions if abs(s["info"]["eye_ts_first"] - s["trials"].start_time.min()) < 500
                and abs(s["info"]["eye_ts_last"] - s["trials"].stop_time.max()) < 500]
print("Sessions with eye and joystick timestamps on the trial clock:", [(s["monkey"], s["session"]) for s in beh_sessions])
signals = {"eye_x": [], "eye_y": [], "joystick": []}
rows = []
t = time.time()
for s in beh_sessions:
    asset = [a for a in mnav.assets() if a.path == s["path"]][0]
    f = mnav._open(asset)
    tr = mnav.pi_trials(s["trials"])
    t0, t1 = mnav.nav_windows(tr)
    for name, key, col in (("eye_x", "eye_position", 0), ("eye_y", "eye_position", 1), ("joystick", "hand_position", None)):
        ts = f[f"processing/behavior/{key}/timestamps"][:]
        d = f[f"processing/behavior/{key}/data"]
        lo, hi = np.searchsorted(ts, t0 - 0.01), np.searchsorted(ts, t1 + 0.01)
        for a, b, x0, x1 in zip(lo, hi, t0, t1):
            seg = d[a:b] if col is None else d[a:b, col]
            grid = np.arange(x0, x1, 0.001)
            if b - a < 10:
                continue
            signals[name].append(np.interp(grid, ts[a:b], seg))
        if name == "joystick":
            # clock check: joystick deflection around the trial table's joystick onset
            on = s["trials"][(s["trials"].trial_type == 3) & (s["trials"].attempt == 1)].joy1onttl.values
            def dev(times, a_, b_):
                l_, h_ = np.searchsorted(ts, times + a_), np.searchsorted(ts, times + b_)
                return np.array([np.abs(d[x:y] - rest).mean() for x, y in zip(l_[:200], h_[:200]) if y > x])
            rest = float(np.median(d[lo[0]:lo[0] + 1000])) if False else None
            pre_seg = np.concatenate([d[x:y] for x, y in zip(np.searchsorted(ts, on[:200] - 0.3), np.searchsorted(ts, on[:200] - 0.05))])
            rest = float(np.median(pre_seg))
            rows.append({"session": s["session"], "trials_used": len(tr),
                         "|dev| before onset": round(float(dev(on, -0.3, -0.05).mean()), 3),
                         "|dev| 0.2-0.45 s after onset": round(float(dev(on, 0.2, 0.45).mean()), 3),
                         "same, onset times + 7 s": round(float(dev(on + 7.0, 0.2, 0.45).mean()), 3)})
print(f"streamed in {time.time() - t:.0f} s")
print(pd.DataFrame(rows).to_string(index=False))
print("Trials per signal:", {k: len(v) for k, v in signals.items()})

In [ ]:
def smooth_trace(x):
    return _g1d(x, mnav.SMOOTH_SD_MS, truncate=mnav.SMOOTH_TRUNCATE, mode="nearest")

def acgs_of(traces):
    out = []
    for x in traces:
        a, _k = mnav.mean_acg([smooth_trace(x)], min_rate=-np.inf)
        if a is not None:
            out.append(a)
    return np.array(out)

def gp_trace(rng, n):
    sig = mnav.GP_LENGTH_MS / np.sqrt(2)
    w = rng.standard_normal(n + 800)
    return _g1d(w, sig, mode="wrap")[400:400 + n]

durations = [len(x) for x in signals["joystick"]]
bank = acgs_of([gp_trace(rng3, n) for n in rng3.choice(durations, 3000)])
lags = mnav.LAGS_METHODS
i650 = int(np.nonzero(lags == 650)[0][0])
rows = []
for name, traces in signals.items():
    A_ = acgs_of(traces)
    n = len(A_)
    for boot in ("a_full_with_replacement", "b_third_without"):
        k = n if boot.startswith("a") else n // 3
        bpi = []
        for _ in range(100):
            idx = rng3.integers(0, n, n) if boot.startswith("a") else rng3.choice(n, k, replace=False)
            bpi.append(mnav.pi_curve(A_[idx].mean(0), lags)[i650])
        bpi = np.array(bpi)
        null = np.array([mnav.pi_curve(bank[rng3.integers(0, len(bank), k)].mean(0), lags)[i650] for _ in range(1000)])
        for chance_name, chance in (("a_mean+2SD", null.mean() + 2 * null.std()), ("b_mean", null.mean())):
            lo_ = stats.ttest_1samp(bpi, chance, alternative="less")
            hi_ = stats.ttest_1samp(bpi, chance, alternative="greater")
            rows.append({"signal": name, "trials": n, "bootstrap": boot, "chance": chance_name, "mean_PI650": round(bpi.mean(), 3),
                         "chance_value": round(chance, 3), "t(99)": round(lo_.statistic, 1), "p_below": lo_.pvalue, "p_above": hi_.pvalue,
                         "below_p<1e-4": bool(lo_.pvalue < 1e-4), "above_p<0.05": bool(hi_.pvalue < 0.05)})
ed3 = pd.DataFrame(rows)
print(ed3.to_string(index=False))
print("\nPaper: mean PI at 650 ms, horizontal eye -0.25, vertical eye -0.44, joystick -0.53 (tolerance 10%).")
print("Readings (bootstrap x chance) with all three signals below chance at p < 1e-4:",
      ed3.groupby(["bootstrap", "chance"])["below_p<1e-4"].all().to_dict())
print("Any signal above chance at p < 0.05 under any reading:", bool(ed3["above_p<0.05"].any()))

**Result: reproduced, in monkey A only.** In the five monkey A sessions whose eye and joystick
traces share the trial clock (1,785 trials), the PI at 650 ms is below chance for all three
signals under every bootstrap and chance reading (one-tailed t-test, p 2.7e-38 to 1.1e-161): mean
PI 0.148 (horizontal eye), 0.136 (vertical eye) and -0.400 (joystick), against chance values of
0.204 to 0.282; no signal is above chance in any reading (one-tailed p = 1.0). The mean PIs
differ from the paper's -0.25, -0.44 and -0.53. Before analysis the joystick trace was checked
against the trial table: its deviation from rest is 0.43 to 0.70 V before the recorded onsets and
1.95 to 2.49 V 0.2 to 0.45 s after them, against 1.08 to 1.32 V at times 7 s later. Monkey M's
traces are on another clock and are not used.

## Summary

| Claim | Outcome |
|---|---|
| fig1bc-produced-matches-actual | reproduced |
| fig1e-immediate-generalization | blocked (data missing) |
| fig2-periodic-proportion | not reproduced (rests on an assumption about the null) |
| fig2d-periodicity-at-065 | partially reproduced (rests on an assumption) |
| ed3-no-eye-hand-periodicity | reproduced (monkey A only) |
| ed2b-pi-navigation-above-iti | blocked (metadata insufficient) |
| ed2c-pi-error-below-correct | partially reproduced (rests on an assumption) |
| fig2e-first-peak-near-065 | partially reproduced |
| fig2f-earlier-peaks-multiples | partially reproduced |
| ed4d-overshoot-longer-period | partially reproduced |
| fig3b-onset-and-offset-coding | reproduced |
| fig3d-offset-coding-stronger | partially reproduced |
| ed4e-rate-codes-distance-over-slope | not reproduced |
| ed4fg-distance-over-landmark | partially reproduced |
| fig3e-xcorr-conserved-iti | blocked (metadata insufficient) |
| ed5-xcorr-conserved-other-epochs | blocked (metadata insufficient) |
| fig4h-periodic-fano-factor | partially reproduced |
| ed8-fano-periodic-without-rate-periodicity | reproduced |
| ed10-motor-preparation-signal | partially reproduced |
| fig4g-reset-model-fits-better | blocked (not attempted in time) |
| fig4b-learned-landmark-input | blocked (not attempted in time) |
| fig4d-model-subscalar-variability | blocked (not attempted in time) |
| ed6d-model-fano-periodic | blocked (not attempted in time) |
| ed5efg-tighter-phase-clustering | blocked (metadata insufficient) |

A failure to reproduce here may be a failure of this attempt, not of the paper: see the
reasoning and the differences from the paper's method.

**How the analysis differs from the paper's.** The dandiset holds 15 of the 32 sessions, so
counts are compared as proportions. The PI null turns the Gaussian process into a firing rate by
exponentiation at the group's median unit rate, which the Methods do not describe, and is built
from a bank of surrogate trials; the classification rule is open and every reading is shown; the
outcomes of fig2d, ed2c and the claims that use periodic units rest on the reading adopted (the
code's score against the surrogate scores). Monkey M's sequence 3 trials are left out wherever
distance is used, as the code does for the PI. All units are used, including those labelled
noise in monkey A.

**What was found about the dataset.** It holds 15 of the 32 sessions, with 1,093 units (364 and
729) against the paper's 1,478; 103 monkey A units are labelled noise. Monkey M's `seqq` 3 trials use vectors in
steps of 0.975 s, which the paper does not describe. The eye and joystick timestamps are on another
clock than the trials in 10 of the 15 sessions. In three sessions (A 09172019, M 04132021, M
04142021) no unit's rate changes around joystick offset, and in M 04132021 the spikes stop halfway
through the trials, so a shared clock could not be established. The ITI is not marked, and the
ITI segments longer than 4 s the paper uses cannot be found. Unit labels differ between monkeys
(`quality` in A, only `KSLabel` in M), and most unit columns have no description.